In [2]:
# Make code imports work in Jupyter Notebook

import sys
import os

# Add the grandparent directory (where parser/ and collector/ live) to sys.path,
# so Python can resolve imports like 'from parser.development_plans.base import ...'
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now use fully qualified imports, which are robust for both scripts and notebooks:
from parser.development_plans.base import BaseDevelopmentPlansParser
from parser.development_plans.boston import BostonDevelopmentPlansParser

import glob
import json
import logging
from pathlib import Path
from typing import Dict, List, Set, Optional

import fitz  # This is pymupdf's import name for the package
import ollama

In [3]:
root_dir = os.path.abspath(os.path.join(os.getcwd(), "../.."))
root_dir

'/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data'

In [4]:


boston_parser = BostonDevelopmentPlansParser(
    model="llama3.2",
    resource_download_directory=os.path.join(root_dir, "downloads/development_plans/boston"),
)
pdf_files_by_folder = boston_parser.read_pdf_directory()


2025-10-12 21:00:24,665 INFO parser.development_plans.boston: Using Ollama with model: llama3.2
2025-10-12 21:00:24,670 INFO parser.development_plans.boston: Ollama connection successful
2025-10-12 21:00:24,679 INFO parser.development_plans.boston: Found PDF files in 297 folders; total 396 PDFs.


models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2025, 10, 12, 17, 48, 31, 628474, tzinfo=TzInfo(-14400)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]
This is the pdf_dir /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston


In [5]:
# Help: Get a flat list of just the PDF file names (not their folder paths)
file_names = []
for files in pdf_files_by_folder.values():
    file_names.extend([os.path.basename(f) for f in files])
file_names
# get the unique file names
unique_file_names = list(set(file_names))
unique_file_names



['Planned_Development_Area__PDA__Application.pdf',
 'Planned_Development_Area__PDA__Amended_and_Restated_Development_Plan.pdf',
 'Letter_of_Intent__LOI_.pdf',
 'Institutional_Master_Plan_Notification_Form__IMPNF_.pdf',
 'Institutional_Master_Plan_Notification_Form-Project_Notification_Form__IMPNF-PNF_.pdf',
 'Planned_Development_Area__PDA__Master_Plan.pdf',
 'Institutional_Master_Plan_Notification_Form__IMPNF__Amendment.pdf',
 'Planned_Development_Area__PDA__Master_Plan_Fact_Sheet.pdf',
 'Planned_Development_Area__PDA__Amended_and_Restated_Master_Plan.pdf',
 'Institutional_Master_Plan_Notification_Form_Project_Notification_Form__IMPNF_PNF_.pdf',
 'Planned_Development_Area__PDA__Master_Plan_Amendment.pdf',
 'BPDA_Board.pdf',
 'Planned_Development_Area__PDA__Amended_and_Restated_Master_Plan_Fact_Sheet.pdf',
 'Planned_Development_Area__PDA__Development_Plan_Fact_Sheet.pdf',
 'Planned_Development_Area__PDA__Development_Plan_Amendment.pdf',
 'Planned_Development_Area__PDA__Development_Plan.

In [6]:
def print_pdfs_matching_text(text: str):
    """
    Find all PDF file names that contain the given text (case-insensitive), 
    print their names and full paths within the download directory.
    """
    # Get unique file names from the previously collected list
    matches = [file for file in unique_file_names if text.lower() in file.lower()]

    if not matches:
        print(f"No files found containing '{text}'")
    else:
        print(f"Found {len(matches)} file(s) matching '{text}':")
        for fname in matches:
            print(f"  - {fname}")
        all_full_paths = []
        for files in pdf_files_by_folder.values():
            for f in files:
                if any(mf in os.path.basename(f) for mf in matches):
                    all_full_paths.append(f)
        print("Full paths to these files:")
        for full_path in all_full_paths:
            print(f"  {full_path}")
        print(f"Found {len(all_full_paths)} file(s) matching '{text}':")
    return all_full_paths

# Example usage:
print_pdfs_matching_text("Letter_of_Intent")

Found 1 file(s) matching 'Letter_of_Intent':
  - Letter_of_Intent__LOI_.pdf
Full paths to these files:
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/251_Causeway_Street/Letter_of_Intent__LOI_.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/716_Columbus_Avenue/Letter_of_Intent__LOI_.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/142-146_St_Marys/Letter_of_Intent__LOI_.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/48-58_Hudson_Street__Parcel_R-1_/Letter_of_Intent__LOI_.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Constitution_Inn__150_Third_Avenue_/Let

['/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/251_Causeway_Street/Letter_of_Intent__LOI_.pdf',
 '/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/716_Columbus_Avenue/Letter_of_Intent__LOI_.pdf',
 '/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/142-146_St_Marys/Letter_of_Intent__LOI_.pdf',
 '/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/48-58_Hudson_Street__Parcel_R-1_/Letter_of_Intent__LOI_.pdf',
 '/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Constitution_Inn__150_Third_Avenue_/Letter_of_Intent__LOI_.pdf',
 '/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Githu

In [7]:
bpda_files = print_pdfs_matching_text("BPDA")

Found 1 file(s) matching 'BPDA':
  - BPDA_Board.pdf
Full paths to these files:
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/295-297_Franklin_Street/BPDA_Board.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/232_A_St__Development_Plan/BPDA_Board.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/3_Aspinwall_Road/BPDA_Board.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/1153_Washington_Street/BPDA_Board.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/10_Malcolm_X_Boulevard/BPDA_Board.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/a

In [12]:
pda_files = print_pdfs_matching_text("Planned_Development_Area__PDA__Master_Plan")

Found 3 file(s) matching 'Planned_Development_Area__PDA__Master_Plan':
  - Planned_Development_Area__PDA__Master_Plan.pdf
  - Planned_Development_Area__PDA__Master_Plan_Fact_Sheet.pdf
  - Planned_Development_Area__PDA__Master_Plan_Amendment.pdf
Full paths to these files:
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/475-511_Dorchester_Avenue__On_the_Dot_/Planned_Development_Area__PDA__Master_Plan.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/475-511_Dorchester_Avenue__On_the_Dot_/Planned_Development_Area__PDA__Master_Plan_Fact_Sheet.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/75-77_Morrissey_Boulevard_Master_Plan/Planned_Development_Area__PDA__Master_Plan.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/

In [13]:
imf_files = print_pdfs_matching_text("Institutional_Master_Plan")

Found 4 file(s) matching 'Institutional_Master_Plan':
  - Institutional_Master_Plan_Notification_Form__IMPNF_.pdf
  - Institutional_Master_Plan_Notification_Form-Project_Notification_Form__IMPNF-PNF_.pdf
  - Institutional_Master_Plan_Notification_Form__IMPNF__Amendment.pdf
  - Institutional_Master_Plan_Notification_Form_Project_Notification_Form__IMPNF_PNF_.pdf
Full paths to these files:
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Brigham_and_Women_s_IMP_2022-2024_Renewal/Institutional_Master_Plan_Notification_Form__IMPNF_.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Wentworth_Institute_of_Technology_IMP_2023-2033/Institutional_Master_Plan_Notification_Form__IMPNF_.pdf
  /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Suffolk_Uni

In [ ]:
spra_docs = print_pdfs_matching_text("Small_Project_Review")

In [ ]:
for file in bpda_files:
    print(file)
    # boston_parser.parse_bpda(file)

2025-10-12 21:00:27,417 INFO parser.development_plans.boston: Found 2 ZONING section(s)


/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/295-297_Franklin_Street/BPDA_Board.pdf
/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/232_A_St__Development_Plan/BPDA_Board.pdf


2025-10-12 21:00:27,708 INFO parser.development_plans.boston: Found 1 ZONING section(s)
2025-10-12 21:00:27,767 INFO parser.development_plans.boston: Found 1 ZONING section(s)
2025-10-12 21:00:27,784 INFO parser.development_plans.boston: Found 1 ZONING section(s)


/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/3_Aspinwall_Road/BPDA_Board.pdf
/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/1153_Washington_Street/BPDA_Board.pdf
/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/10_Malcolm_X_Boulevard/BPDA_Board.pdf


2025-10-12 21:00:27,947 INFO parser.development_plans.boston: Found 1 ZONING section(s)
2025-10-12 21:00:28,041 INFO parser.development_plans.boston: Found 1 ZONING section(s)
2025-10-12 21:00:28,115 INFO parser.development_plans.boston: Found 1 ZONING section(s)


...

2025-10-12 21:00:35,327 WARNING parser.development_plans.boston: No ZONING sections found in /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Boston_Children_s_Hospital_IMP_2023-2025/BPDA_Board.pdf
2025-10-12 21:00:35,438 WARNING parser.development_plans.boston: No ZONING sections found in /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/259-267_Summer_Street/BPDA_Board.pdf
2025-10-12 21:00:35,522 WARNING parser.development_plans.boston: No ZONING sections found in /Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Seaport_Square_Block_L3_L6/BPDA_Board.pdf


/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/259-267_Summer_Street/BPDA_Board.pdf
/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/Seaport_Square_Block_L3_L6/BPDA_Board.pdf
/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/554-562_Columbia_Road/BPDA_Board.pdf


2025-10-12 21:00:35,583 INFO parser.development_plans.boston: Found 1 ZONING section(s)


/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/data/downloads/development_plans/boston/pdfs/134_Hampden_Street/BPDA_Board.pdf


KeyboardInterrupt: 